> ### **Sorting Algorithms**

> *Created by Jason Naicker*.

> *[Github](https://github.com/JasonNaicker)*

---

In [76]:
import sys, os, gc

from typing import Final, Optional, override, Final
from abc import abstractmethod, ABC
from time import perf_counter
from functools import wraps
import random
from enum import Enum

In [77]:
class SearchingTemplate(ABC):
    def __init__(self, items : list[int]) -> None:
        self._original_items = items[:]
        self._items = self._original_items[:]
        self._size = len(self._items)

    @abstractmethod
    def search(self) -> any:
        '''Implement Searching algorithm'''
        pass

In [78]:
#Benchmarker

> #### **Algos**

In [79]:
class SelectionSearch(SearchingTemplate):
    @override
    def search(self, value : int) -> int:
        for idx, val in enumerate(self._items):
            if val == value:
                return idx
        
        return -1

In [80]:
class BinarySearch(SearchingTemplate):
    @override
    def search(self, value : int) -> int:
        sorted_list = sorted(self._items)
        left_idx : int = 0
        right_idx : int = len(sorted_list) - 1

        while(left_idx <= right_idx):
            middle_idx : int = left_idx + (right_idx - left_idx) // 2
            middle_value = sorted_list[middle_idx]

            if value < middle_value:
                right_idx = middle_idx - 1
            elif value > middle_value:
                left_idx = middle_idx + 1
            elif value == middle_value:
                return middle_idx
            else:
                return -1

In [81]:
class MedianOfMedians:
    def select_pivot(self, items: list[int], start: int, end: int):
        n = end - start + 1

        if n <= 5:
            group = sorted(items[start:end+1])
            return group[len(group)//2]

        medians = []
        for i in range(start, end+1, 5):
            sub_end = min(i + 4, end)
            group = sorted(items[i:sub_end+1])
            medians.append(group[len(group)//2])

        return self.select_pivot(medians, 0, len(medians)-1)

In [82]:
class PivotStrategy(Enum):
    MEDIAN_OF_MEDIANS = "Median_Of_Medians"
    RANDOM = "Random"

class Order(Enum):
    LARGEST = "Largest"
    SMALLEST = "Smallest"

class QuickSelect(SearchingTemplate):
    def _partition(self, start : int, end : int, pivot_index : int):
        pivot_value = self._items[pivot_index]
        self._items[pivot_index], self._items[end] = self._items[end], self._items[pivot_index]
        store_index = start

        for i in range(start, end):
            if self._items[i] < pivot_value:
                self._items[i], self._items[store_index] = self._items[store_index], self._items[i]
                store_index += 1
        self._items[store_index], self._items[end] = self._items[end], self._items[store_index]
        return store_index

    def _QuickSelect(self, start : int, end : int, k, pivot, order):
        if start == end:
            return self._items[start]
        
        if pivot == PivotStrategy.MEDIAN_OF_MEDIANS:
            median = MedianOfMedians()
            pivot_index = median.select_pivot(self._items, start, end)
        else:
            pivot_index = random.randint(start, end)

        pivot_index = self._partition(start, end, pivot_index)

        if k == pivot_index:
            return self._items[pivot_index]
        elif k < pivot_index:
            return self._QuickSelect(start, pivot_index - 1, k, pivot, order)
        else:
            return self._QuickSelect(pivot_index + 1, end, k, pivot, order)

    @override
    def search(self, k : int, order : Order = Order.SMALLEST, pivot : PivotStrategy = PivotStrategy.RANDOM):
        if order == Order.SMALLEST:
            k = k - 1
        elif order == Order.LARGEST:
            k = self._size - k
        else:
            raise ValueError("Order can only be Smallest or Largest")
        
        if k < 0 or k >= self._size:
            raise ValueError("K out of range")
        return self._QuickSelect(0, (self._size - 1), k, pivot=pivot, order=order)


In [85]:
a = QuickSelect([1,5,4,6,2,8,3])
print(a.search(1, Order.LARGEST, PivotStrategy.RANDOM))


8
